In [ ]:
!pip install mne

     |████████████████████████████████| 6.6MB 25kB/s 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [ ]:
def band_pass_filter(eeg, freq_range):
  info = mne.create_info(22, 250, ch_types=["eeg"] * 22)
  raw = mne.io.RawArray(eeg.T, info)
  raw.filter(freq_range[0], freq_range[1], fir_design='firwin')

  return raw._data.T

In [ ]:
def load_data(mode='train', fno = 1):
  if (mode=='train'):
    fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
    file_data = loadmat(fname)
    data = file_data['data']
    df = pd.DataFrame()
    for i in range(0, 6):
        idx = 3+i
        pos_data = data[0][idx][0][0][1]
        label_data = data[0][idx][0][0][2]
        temp = pd.DataFrame(data[0][idx][0][0][0])
        label = np.zeros(len(temp))
        count = 0
        for j in pos_data:
            label[j] = label_data[count]
            count += 1
        temp['class'] = label
        df = pd.concat([df, temp], ignore_index=True)

  elif (mode=='test'):
    fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
    file_data = loadmat(fname)
    data = file_data['data']
    df = pd.DataFrame()
    for i in range(0, 6):
        idx = 3+i
        pos_data = data[0][idx][0][0][1]
        label_data = data[0][idx][0][0][2]
        temp = pd.DataFrame(data[0][idx][0][0][0])
        label = np.zeros(len(temp))
        count = 0
        for j in pos_data:
            label[j] = label_data[count]
            count += 1
        temp['class'] = label
        df = pd.concat([df, temp], ignore_index=True)
    
  return df

In [ ]:
def drop_classes(df):
  i = 0
  indexes_to_drop = []
  while i < len(df):
    if(df['class'][i]==4):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    elif(df['class'][i]==3):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    else:
      i += 1

  indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
  df_sliced = df.take(list(indexes_to_keep))

  df_sliced = df_sliced.reset_index(drop=True)
  return df_sliced

In [ ]:
def fbcsp(df, sfreq, train=True, csp_objects=None):
  freq = 4
  increment = 4
  end_freq = 40
  event_dict = {'Left/Hands': 1, 'Right/Hands': 2}
  if train==True:
    csp_objects = []
    csp_data = []
    while freq < end_freq:
      freq_range = []
      freq_range.append(freq)
      freq_range.append(freq+increment)
      freq += increment
      out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

      df_new = pd.DataFrame(out_data)
      df_new['label'] = df.iloc[:, -1].values

      info = mne.create_info(23, sfreq, ch_types=["eeg"] * 22 + ['stim'] * 1)
      raw = mne.io.RawArray(df_new.T, info)

      events = mne.find_events(raw, stim_channel='22')
      picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False,
                   exclude='bads')
      epochs = mne.Epochs(raw, events, event_id=event_dict, tmin=-1, tmax=4, picks=picks, preload=True)
      epochs = epochs.crop(tmin=0.5, tmax=2.5)
      y = epochs.events[:, -1]
      X = epochs.get_data()

      csp = CSP(n_components=2, reg=None, log=True, norm_trace=False)

      final_data = csp.fit_transform(X, y)

      csp_objects.append(csp)
      csp_data.append(final_data)

    return np.array(csp_objects), np.array(csp_data), y
  
  else:
    csp_data = []
    count = 0
    while freq < end_freq:
      freq_range = []
      freq_range.append(freq)
      freq_range.append(freq+increment)
      freq += increment
      out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

      df_new = pd.DataFrame(out_data)
      df_new['label'] = df.iloc[:, -1].values

      info = mne.create_info(23, sfreq, ch_types=["eeg"] * 22 + ['stim'] * 1)
      raw = mne.io.RawArray(df_new.T, info)

      events = mne.find_events(raw, stim_channel='22')
      picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False,
                   exclude='bads')
      epochs = mne.Epochs(raw, events, event_id=event_dict, tmin=-0.1, tmax=2, picks=picks, preload=True)
      epochs = epochs.crop(tmin=0.0, tmax=2.0)
      y = epochs.events[:, -1]
      X = epochs.get_data()

      final_data = csp_objects[count].transform(X)
      count += 1

      csp_data.append(final_data)

    return np.array(csp_data), y

  # return np.array(csp_objects), np.array(csp_data), y

In [ ]:
def itr(n_class, p_class, c_time):
  B = (np.log2(n_class) + (p_class * np.log2(p_class)) + ((1-p_class) * np.log2((1-p_class)/(n_class-1)))) / c_time * 60

  return B

def performance_metrics(y_test, y_pred):
  acc = accuracy_score(y_test, y_pred)
  print('Accuracy Score: ', acc)
  print('Cohen Kappa Score: ', cohen_kappa_score(y_test, y_pred))
  print('ITR (bits per minute): ', itr(n_class=2, p_class=acc, c_time=2))
  print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))

In [ ]:
def get_train_data(fno):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}

  df = load_data(mode='train', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)

  csp_objects, csp_data, y = fbcsp(train_df, sfreq, train=True)

  final_data = pd.DataFrame(csp_data[0])
  col_count = 4
  for i in range(1, len(csp_data)):
    for j in range(len(csp_data[i].T)):
      final_data[str(col_count)] = csp_data[i].T[j]
      col_count += 1

  return final_data.values, y, csp_objects

def get_eval_data(fno, csp_objects):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}

  df = load_data(mode='test', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)
  csp_data, y = fbcsp(train_df, sfreq, train=False, csp_objects=csp_objects)

  final_data = pd.DataFrame(csp_data[0])
  col_count = 4
  for i in range(1, len(csp_data)):
    for j in range(len(csp_data[i].T)):
      final_data[str(col_count)] = csp_data[i].T[j]
      col_count += 1

  return final_data.values, y

In [ ]:
def train_mibif(X, y):
  print('------Train---------')
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

  #get the best k features base on MIBIF algorithm
  select_K = SelectKBest(mutual_info_classif,k=8).fit(X, y)
  extra = select_K.get_support()
  if extra[0] == True:
    extra[1] = True
  for i in range(2, len(extra)):
    if extra[i] == True:
      if i%2 == 0:
        extra[i+1] = True
      else:
        extra[i-1] = True
  pos = np.where(extra==False)
  New_train = np.delete(X_train, list(pos[0]), 1)
  New_test = np.delete(X_test, list(pos[0]), 1)
  # New_train=select_K.transform(X_train)
  # New_test=select_K.transform(X_test)
  ss = StandardScaler()
  New_train = ss.fit_transform(New_train,y_train)
  New_test = ss.transform(New_test)

  print('####### SVM#####')
  svm = SVC()
  svm.fit(New_train, y_train)
  y_pred = svm.predict(New_test)
  performance_metrics(y_test, y_pred)
  
  print('##########LDA#########')
  lda = LinearDiscriminantAnalysis()
  lda.fit(New_train, y_train)
  y_pred = lda.predict(New_test)
  performance_metrics(y_test, y_pred)

  return list(pos[0]), svm, lda, ss

def eval_mibif(pos, svm, lda, ss, X, y):
  print('------Test--------')
  X = np.delete(X, pos, 1)
  X = ss.transform(X)
  print('#####SVM######')
  y_pred = svm.predict(X)
  performance_metrics(y, y_pred)
  print('#####LDA######')
  y_pred = lda.predict(X)
  performance_metrics(y, y_pred)


In [ ]:
def train_rf(X, y):
  print('--------Train--------')
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
  clf = RandomForestClassifier(random_state=42)
  ss = StandardScaler()
  X_train = ss.fit_transform(X_train, y_train)
  clf.fit(X_train, y_train)
  X_test = ss.transform(X_test)
  y_pred = clf.predict(X_test)
  ####Random Forest#######
  performance_metrics(y_test, y_pred)

  return clf, ss
def eval_rf(clf, ss, X, y):
  print('--------Test------')
  X = ss.transform(X)
  y_pred = clf.predict(X)
  performance_metrics(y, y_pred)

# Subject A01

In [ ]:
X, y, csp_objects = get_train_data(fno=1)
X_eval, y_eval = get_eval_data(fno=1,csp_objects=csp_objects)

In [ ]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[18  0]
 [ 0 18]]
##########LDA#########
Accuracy Score:  0.9444444444444444
Cohen Kappa Score:  0.8888888888888888
ITR (bits per minute):  20.713697125490242
Confusion Matrix:  [[17  1]
 [ 1 17]]
------Test--------
#####SVM######
Accuracy Score:  0.6319444444444444
Cohen Kappa Score:  0.26388888888888884
ITR (bits per minute):  1.5249783629662372
Confusion Matrix:  [[56 16]
 [37 35]]
#####LDA######
Accuracy Score:  0.5902777777777778
Cohen Kappa Score:  0.18055555555555558
ITR (bits per minute):  0.7093685992417309
Confusion Matrix:  [[58 14]
 [45 27]]


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  


In [ ]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.9444444444444444
Cohen Kappa Score:  0.8888888888888888
ITR (bits per minute):  20.713697125490242
Confusion Matrix:  [[17  1]
 [ 1 17]]
--------Test------
Accuracy Score:  0.5763888888888888
Cohen Kappa Score:  0.1527777777777778
ITR (bits per minute):  0.5070937886328808
Confusion Matrix:  [[47 25]
 [36 36]]


# Subject A02

In [ ]:
X, y, csp_objects = get_train_data(fno=2)
X_eval, y_eval = get_eval_data(fno=2,csp_objects=csp_objects)

In [ ]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.9444444444444444
Cohen Kappa Score:  0.8888888888888888
ITR (bits per minute):  20.713697125490242
Confusion Matrix:  [[18  0]
 [ 2 16]]
##########LDA#########
Accuracy Score:  0.9166666666666666
Cohen Kappa Score:  0.8333333333333334
ITR (bits per minute):  17.585494490890987
Confusion Matrix:  [[17  1]
 [ 2 16]]
------Test--------
#####SVM######
Accuracy Score:  0.4652777777777778
Cohen Kappa Score:  -0.06944444444444442
ITR (bits per minute):  0.10444566385107823
Confusion Matrix:  [[22 50]
 [27 45]]
#####LDA######
Accuracy Score:  0.5069444444444444
Cohen Kappa Score:  0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[30 42]
 [29 43]]


In [ ]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.9166666666666666
Cohen Kappa Score:  0.8333333333333334
ITR (bits per minute):  17.585494490890987
Confusion Matrix:  [[17  1]
 [ 2 16]]
--------Test------
Accuracy Score:  0.5
Cohen Kappa Score:  0.0
ITR (bits per minute):  0.0
Confusion Matrix:  [[18 54]
 [18 54]]


# Subject A03

In [ ]:
X, y, csp_objects = get_train_data(fno=3)
X_eval, y_eval = get_eval_data(fno=3,csp_objects=csp_objects)

In [ ]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[18  0]
 [ 0 18]]
##########LDA#########
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[18  0]
 [ 0 18]]
------Test--------
#####SVM######
Accuracy Score:  0.5069444444444444
Cohen Kappa Score:  0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[ 9 63]
 [ 8 64]]
#####LDA######
Accuracy Score:  0.5138888888888888
Cohen Kappa Score:  0.02777777777777779
ITR (bits per minute):  0.016700007291030605
Confusion Matrix:  [[13 59]
 [11 61]]


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  


In [ ]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[18  0]
 [ 0 18]]
--------Test------


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  


Accuracy Score:  0.5625
Cohen Kappa Score:  0.125
ITR (bits per minute):  0.3390177513450765
Confusion Matrix:  [[23 49]
 [14 58]]


# Subject A05 and A06

In [ ]:
X, y, csp_objects = get_train_data(fno=5)
X_eval, y_eval = get_eval_data(fno=5,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

In [ ]:
X, y, csp_objects = get_train_data(fno=6)
X_eval, y_eval = get_eval_data(fno=6,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events

<ipython-input-14-14a7b8d9f9f7>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 12.00 Hz
- Upper transition bandwidth: 3.00 Hz (-6 dB cutoff frequency: 13.50 Hz)
- Filter length: 413 samples (1.652 sec)

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mea

<ipython-input-14-14a7b8d9f9f7>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 12.00
- Lower transition bandwidth: 3.00 Hz (-6 dB cutoff frequency: 10.50 Hz)
- Upper passband edge: 16.00 Hz
- Upper transition bandwidth: 4.00 Hz (-6 dB cutoff frequency: 18.00 Hz)
- Filter length: 275 samples (1.100 sec)

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 proj

...

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 36.00
- Lower transition bandwidth: 9.00 Hz (-6 dB cutoff frequency: 31.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff f

<ipython-input-14-14a7b8d9f9f7>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.9444444444444444
Cohen Kappa Score:  0.8888888888888888
ITR (bits per minute):  20.713697125490242
Confusion Matrix:  [[18  0]
 [ 2 16]]
##########LDA#########
Accuracy Score:  0.9166666666666666
Cohen Kappa Score:  0.8333333333333334
ITR (bits per minute):  17.585494490890987
Confusion Matrix:  [[16  2]
 [ 1 17]]
------Test--------
#####SVM######
Accuracy Score:  0.5
Cohen Kappa Score:  0.0
ITR (bits per minute):  0.0
Confusion Matrix:  [[42 30]
 [42 30]]
#####LDA######
Accuracy Score:  0.5069444444444444
Cohen Kappa Score:  0.01388888888888884
ITR (bits per minute):  0.

<ipython-input-14-14a7b8d9f9f7>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.9444444444444444
Cohen Kappa Score:  0.8888888888888888
ITR (bits per minute):  20.713697125490242
Confusion Matrix:  [[18  0]
 [ 2 16]]
--------Test------
Accuracy Score:  0.5
Cohen Kappa Score:  0.0
ITR (bits per minute):  0.0
Confusion Matrix:  [[51 21]
 [51 21]]
